# Bộ song ngữ thứ hai (Tiki): mô hình B dịch Việt sang Trung và CometKiwi chấm điểm trên Kaggle (bước 4 cho mô hình B, bước 6a cho A và B)

Notebook này làm phần cần GPU: **dịch tiếng Việt sang tiếng Trung giản thể** (thuộc tính sản phẩm Tiki) bằng **mô hình B** (dùng cả hai GPU; chọn ở dòng `MO_HINH_B` của ô Cấu hình: từ 21/10/2026 mặc định là Qwen3-30B-A3B-Instruct-2507 bản nén 4-bit, thay HY-MT1.5-7B vì giấy phép HY-MT hạn chế dùng đầu ra), rồi **chấm điểm CometKiwi** cho cả hai bản dịch (song song, mỗi bản một GPU). **Mô hình A (Gemma 4) KHÔNG dịch ở đây**: nó dịch ở notebook riêng `song-ngu-2-gemma-kaggle` (chỉ gọi API, không GPU). Ở đây chỉ lấy đầu ra của notebook Gemma để chấm điểm. Mô hình A cũ (DeepSeek trả phí) mặc định TẮT (`CHAY_DICH_A = False`) nên notebook này không gọi API trả phí nào. Các bước khác chạy trên laptop (`scripts/song_ngu_2/`).

**Cách chạy để không mất kết quả (đóng tab, tắt máy vẫn chạy)**
1. Settings: Accelerator **GPU T4 x2**, Internet **On**. Add-ons > Secrets: thêm secret `HF_TOKEN` (token đọc của Hugging Face, đã chấp nhận điều khoản `Unbabel/wmt22-cometkiwi-da`), bật Attach. KHÔNG cần khoá API nào khác (không dán khoá vào notebook hay chat).
2. Add Data: Dataset riêng tư của bộ Tiki, chứa thư mục `vao_kaggle` của `data/processed/song_ngu_2/` (có `don_vi_can_dich.parquet`, `dich.py`, `cham_kiwi.py`, `cau_hinh_kaggle.json` với `huong = vi_zh`). Không gắn Dataset của bộ 1 cùng lúc.
3. Sửa ô "Cấu hình" (chạy thử hay chạy thật), rồi bấm **Save Version > Save & Run All (Commit)**. Kaggle chạy nền trên máy của Kaggle; đóng trình duyệt hay tắt máy vẫn chạy.
4. Kết quả nằm ở tab **Output** của phiên bản đó (thư mục `ra_tiki/` hoặc `ra_tiki_thu/`). Tải về laptop ngay khi xong.

**Bị ngắt hoặc hết giờ thì sao:** mỗi lô dịch xong được ghi ngay thành file. Script tự dừng êm sau `GIO_TOI_DA_DICH` để Kaggle kịp lưu đầu ra. Muốn làm tiếp: tạo phiên bản mới, **Add Data > Notebook Output** của phiên bản trước (đầu ra HY-MT), rồi Save & Run All lại; script đọc phần đã có và chỉ dịch phần còn thiếu. **Để chấm điểm cho bản Gemma:** khi notebook Gemma đã dịch xong cả bộ, thêm **Notebook Output của notebook Gemma** vào Add Data của phiên bản cuối (hoặc phiên bản chạy riêng chấm điểm); ô chấm điểm tự chép bản Gemma sang rồi chấm. Không chạy cả bộ ở chế độ tương tác (bấm từng ô), vì thư mục làm việc mất khi hết phiên.

In [ ]:
# ===== Cấu hình (chỉ sửa ô này) =====
HUONG = "vi_zh"           # chiều dịch: Việt sang Trung (bộ Tiki); không đổi
CHE_DO = "thu"            # "thu": thử khói (mốc M0, khoảng 200 đơn vị mỗi mô hình) | "that": chạy cả bộ
SO_DON_VI_THU = 200
GIO_TOI_DA_DICH = 7.5     # giờ cho phần dịch; hết giờ thì dừng êm, chạy phiên bản sau để làm tiếp. 7,5 + 1 (chấm điểm) + cài đặt < 9 giờ: an toàn dù Kaggle giới hạn phiên 9 hay 12 giờ
GIO_TOI_DA_KIWI = 1.0     # giờ cho phần chấm điểm
GIO_TOI_DA_THU = 1.0      # chế độ thử: phần dịch tối đa 1 giờ
# API DeepSeek chính thức là TRẢ PHÍ. Hai ngân sách dưới đây là mức TỐI ĐA (USD): chạm là dừng, không tốn thêm.
# Ước một lần chạy cả bộ khoảng 2 đến 5 USD; chạy thử 200 đơn vị khoảng 0,01 USD.
NGAN_SACH_USD_THU = 0.5
NGAN_SACH_USD_THAT = 8.0
SONG_SONG_API = 16        # số yêu cầu gọi API cùng lúc; gặp lỗi 429 thì giảm xuống 4
# Địa chỉ gọi API của Alibaba Cloud: lấy đúng trong bảng điều khiển Model Studio (nút xem mã gọi), dạng
# https://<WorkspaceId>.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1 . Để trống thì dùng địa chỉ mặc định trong dich.py (chưa kiểm).
API_BASE_URL = ""
API_EXTRA = ""            # để trống: dùng cấu hình mô hình (tự thử các cách tắt suy luận). Chỉ điền JSON nếu cần ghi đè
BACKEND = "auto"          # "auto": thử vLLM, lỗi thì dùng transformers 4-bit (rất chậm) | "vllm" | "hf"
GOI_VLLM = "vllm"         # lần thử M0 (09/10/2026) vLLM 0.31.0 chạy được trên T4
CHAM_KIWI = True
CHAY_DICH_A = False       # False: KHÔNG gọi API dịch mô hình A (A = Gemma chạy ở notebook riêng; True chỉ để chạy lại cách DeepSeek trả phí cũ)
MO_HINH_A = "api_tuy_chon"   # tên thư mục bản dịch của mô hình A (Gemma qua notebook song-ngu-1-gemma-kaggle)
MO_HINH_B = "qwen3_14b"  # mô hình B: "qwen3_30b_a3b" (Qwen3-30B-A3B-Instruct-2507, AWQ 4-bit cộng đồng) | "qwen3_14b" (Qwen3-14B, AWQ chính thức) | "hy_mt15_7b" (cũ, vướng giấy phép)

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path

def _huong(p):
    try:
        return json.loads((Path(p).parent / "cau_hinh_kaggle.json").read_text(encoding="utf-8")).get("huong")
    except Exception:
        return None

ds = [p for p in glob.glob("/kaggle/input/**/don_vi_can_dich.parquet", recursive=True) if _huong(p) == "vi_zh"]
assert ds, "Chưa gắn Dataset của bộ Tiki (don_vi_can_dich.parquet cùng cau_hinh_kaggle.json có huong = vi_zh) (Add Data)."
VAO = Path(ds[0]).parent
MO_HINH = json.loads((VAO / "cau_hinh_kaggle.json").read_text(encoding="utf-8"))["mo_hinh"]
MO_HINH["A"] = MO_HINH_A          # A là Gemma (api_tuy_chon), không phải DeepSeek trong tệp cấu hình cũ
MO_HINH["B"] = MO_HINH_B          # B chọn ở ô Cấu hình, không theo tệp cấu hình trong Dataset
RA = Path("/kaggle/working/ra_tiki" if CHE_DO == "that" else "/kaggle/working/ra_tiki_thu")
RA.mkdir(parents=True, exist_ok=True)
# đầu ra phiên bản trước (Add Data > Notebook Output): thư mục chứa <mo_hinh>/lo_*.parquet
CU = [] if CHE_DO == "thu" else sorted({str(Path(p).parent.parent) for p in glob.glob("/kaggle/input/**/lo_*.parquet", recursive=True)
                                        if Path(p).parent.parent.name == "ra_tiki"})   # chỉ lấy đầu ra chạy thật, không lẫn ra_tiki_thu
(RA / "kiem_luu.txt").write_text(f"File thử lưu, ghi lúc {time.strftime('%Y-%m-%d %H:%M:%S')}. Thấy file này ở tab Output là đầu ra đã được giữ.\n",
                                 encoding="utf-8")
so_gpu = len([l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if l.startswith("GPU")])
print("Dữ liệu vào:", VAO)
print("Thư mục ra:", RA)
print("Đầu ra cũ dùng để chạy tiếp:", CU or "không có")
# bản dịch của A (Gemma) lấy từ Notebook Output của notebook Gemma (thư mục ra_gemma_tiki) hoặc từ đầu ra cũ của chính notebook này (thư mục ra)
DIR_A = [] if CHE_DO == "thu" else sorted({str(Path(p).parent.parent) for p in glob.glob(f"/kaggle/input/**/{MO_HINH_A}/lo_*.parquet", recursive=True)
                                           if Path(p).parent.parent.name in ("ra_gemma_tiki", "ra_tiki")})
print("Nơi có bản dịch của A để chấm điểm:", DIR_A or "chưa có (chưa gắn Notebook Output của notebook Gemma)")
print("Mô hình:", MO_HINH, "| số GPU:", so_gpu)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -n 2

In [ ]:
# Khoá từ Kaggle Secrets (không in ra, không ghi vào file)
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    for _ten in (["HF_TOKEN", "DASHSCOPE_API_KEY"] if CHAY_DICH_A else ["HF_TOKEN"]):
        try:
            os.environ[_ten] = _us.get_secret(_ten)
            print(f"Đã nạp {_ten} từ Kaggle Secrets.")
        except Exception as e:
            print(f"Chưa có secret {_ten} ({type(e).__name__}).", {"HF_TOKEN": "Mô hình B vẫn tải được; CometKiwi sẽ không tải được.",
                  "DASHSCOPE_API_KEY": "Mô hình A (DeepSeek qua Alibaba Cloud) sẽ bị bỏ qua."}[_ten])
    if os.environ.get("DASHSCOPE_API_KEY", "").startswith("sk-sp-"):
        os.environ.pop("DASHSCOPE_API_KEY")
        print("!! Khoá bắt đầu bằng sk-sp- là khoá Token Plan: điều khoản cấm chạy tự động và gọi hàng loạt (có thể bị khoá). "
              "Đã bỏ qua mô hình A. Dùng khoá của API trả theo lượng dùng (pay-as-you-go).")
except Exception as e:
    print("Không đọc được Kaggle Secrets:", type(e).__name__)
# Mô hình và gói tải vào /tmp (không vào /kaggle/working, để đầu ra không bị phình)
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["UV_CACHE_DIR"] = "/tmp/uv_cache"

In [ ]:
# Môi trường riêng cho phần dịch (Python 3.11, cài bằng uv), không đụng gói có sẵn của Kaggle
!pip -q install uv
!uv venv /tmp/env_dich --python 3.11 -q
!uv pip install -q --python /tmp/env_dich/bin/python pandas pyarrow huggingface_hub "transformers>=4.56" accelerate bitsandbytes torch
if BACKEND != "hf":
    !uv pip install -q --python /tmp/env_dich/bin/python "{GOI_VLLM}" || echo "CÀI vLLM LỖI: sẽ dùng transformers"
!/tmp/env_dich/bin/python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU', torch.cuda.device_count())"
!/tmp/env_dich/bin/python -c "import vllm; print('vllm', vllm.__version__)" || echo "không có vLLM"

In [ ]:
def bat(lenh, gpu, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, PYTHONUNBUFFERED="1")
    return subprocess.Popen(lenh, stdout=open(log, "w"), stderr=subprocess.STDOUT, env=env)

def cho(ps, logs):
    # chờ các tiến trình xong, in dòng cuối của log mỗi 2 phút
    while any(p.poll() is None for p in ps):
        time.sleep(120)
        for log in logs:
            dong = Path(log).read_text(errors="ignore").strip().splitlines()
            print(f"[{Path(log).name}]", dong[-1][:220] if dong else "")
    for p, log in zip(ps, logs):
        print(f"--- {Path(log).name}: mã thoát {p.returncode}, 15 dòng cuối ---")
        print("\n".join(Path(log).read_text(errors="ignore").strip().splitlines()[-15:]))
    return [p.returncode for p in ps]

def chay_song_song(viec, ten_log):
    # viec: danh sách (lệnh, gpu). Song song nếu có >= 2 GPU, không thì lần lượt trên GPU 0.
    logs = [str(RA / f"{ten_log}_{i}.log") for i in range(len(viec))]
    if so_gpu >= 2:
        return cho([bat(l, str(g), log) for (l, g), log in zip(viec, logs)], logs)
    return sum([cho([bat(l, "0", log)], [log]) for (l, _), log in zip(viec, logs)], [])

# Dịch: B (MO_HINH_B, dùng cả hai GPU); A (DeepSeek cũ, gọi API) chỉ chạy khi CHAY_DICH_A = True, cùng lúc với B.
PY = "/tmp/env_dich/bin/python"
thu = str(SO_DON_VI_THU if CHE_DO == "thu" else 0)
cu = sum([["--cu", c] for c in CU], [])
tat_ca_gpu = ",".join(str(i) for i in range(max(so_gpu, 1)))

gio_dich = min(GIO_TOI_DA_DICH, GIO_TOI_DA_THU) if CHE_DO == "thu" else GIO_TOI_DA_DICH

def lenh_dich(k, them):
    return ([PY, str(VAO / "dich.py"), "--vao", str(VAO / "don_vi_can_dich.parquet"), "--ra", str(RA), "--mo_hinh", MO_HINH[k], "--huong", HUONG,
             "--backend", BACKEND, "--gio_toi_da", str(gio_dich), "--thu", thu] + them + cu)

# A và B chạy song song: A gọi thử API ngay đầu script, hỏng thì A tự dừng, B vẫn chạy.
# Quá gio_dich thì A ngừng gửi yêu cầu mới kể cả đang giữa lô; lô nhỏ để kết quả được ghi dần.
viec = []
if CHAY_DICH_A and os.environ.get("DASHSCOPE_API_KEY"):
    them_a = ["--song_song", str(SONG_SONG_API), "--lo", "100" if CHE_DO == "thu" else "400",
              "--ngan_sach_usd", str(NGAN_SACH_USD_THU if CHE_DO == "thu" else NGAN_SACH_USD_THAT)] + \
             (["--api_extra", API_EXTRA] if API_EXTRA.strip() else []) + \
             (["--api_base_url", API_BASE_URL] if API_BASE_URL.strip() else [])
    viec.append(("A", lenh_dich("A", them_a), ""))      # CUDA_VISIBLE_DEVICES rỗng: không đụng GPU
else:
    print("Không dịch mô hình A ở notebook này (CHAY_DICH_A = False hoặc chưa có khoá): A = Gemma dịch ở notebook song-ngu-1-gemma-kaggle.")
viec.append(("B", lenh_dich("B", []), tat_ca_gpu))
logs = [str(RA / f"log_dich_{k}.log") for k, _, _ in viec]
t = time.time()
ma_dich = cho([bat(l, g, log) for (k, l, g), log in zip(viec, logs)], logs)
print(f"Phần dịch xong sau {(time.time() - t) / 60:.0f} phút; mã thoát {dict(zip([k for k, _, _ in viec], ma_dich))}")
# xoá bản mô hình đã dùng khỏi /tmp (bản đầy đủ khoảng 15-16 GB) để đủ chỗ cho CometKiwi
import shutil
for d in glob.glob("/tmp/hf_cache/hub/models--*"):
    if "cometkiwi" not in d:
        shutil.rmtree(d, ignore_errors=True)
!df -h /tmp | tail -n 1

In [ ]:
# Bước 6a: chấm điểm CometKiwi trong môi trường riêng (gói của COMET dễ đụng gói của vLLM)
import shutil
import pandas as pd
# chép bản dịch của A (Gemma) vào thư mục ra của phiên này để cham_kiwi.py đọc (nó chỉ đọc <ra>/<mô hình>/lo_*.parquet)
(RA / MO_HINH["A"]).mkdir(parents=True, exist_ok=True)
for _k, _d in enumerate(DIR_A):
    for _f in sorted(Path(_d).glob(f"{MO_HINH['A']}/lo_*.parquet")):
        _dich = RA / MO_HINH["A"] / f"lo_a{_k}_{_f.name}"
        if not _dich.exists():
            shutil.copy2(_f, _dich)
print("Bản dịch A (Gemma) sẵn sàng để chấm:", sum(len(pd.read_parquet(f, columns=["ma_don_vi"])) for f in (RA / MO_HINH["A"]).glob("lo_*.parquet")) if list((RA / MO_HINH["A"]).glob("lo_*.parquet")) else 0, "đơn vị")
if CHAM_KIWI and os.environ.get("HF_TOKEN"):
    !uv venv /tmp/env_kiwi --python 3.11 -q
    # unbabel-comet 2.2.7 đòi numpy<2; pyarrow bản mới không nạp được với numpy 1.x (lỗi ở lần thử M0) nên ghim bộ bản đi được với nhau
    !uv pip install -q --python /tmp/env_kiwi/bin/python unbabel-comet "numpy<2" "pandas>=2.1,<2.3" "pyarrow>=15,<18" "setuptools<81"
    PYK = "/tmp/env_kiwi/bin/python"
    !{PYK} -c "import numpy, pandas, pyarrow, comet; print('numpy', numpy.__version__, '| pandas', pandas.__version__, '| pyarrow', pyarrow.__version__)"
    viec_kiwi = [([PYK, str(VAO / "cham_kiwi.py"), "--vao", str(VAO / "don_vi_can_dich.parquet"), "--ra", str(RA),
                   "--mo_hinh", MO_HINH[k], "--gio_toi_da", str(GIO_TOI_DA_KIWI)] + cu, gpu) for gpu, k in enumerate(["A", "B"])]
    t = time.time()
    ma_kiwi = chay_song_song(viec_kiwi, "log_kiwi")
    print(f"Phần chấm điểm xong sau {(time.time() - t) / 60:.0f} phút")
else:
    print("Bỏ qua CometKiwi (CHAM_KIWI = False hoặc chưa có HF_TOKEN). Có thể chạy phiên bản sau để chấm riêng.")

In [ ]:
# Tóm tắt (mốc M0: xem backend nào chạy được, tốc độ thật, và ước thời gian cả bộ)
import pandas as pd
tong = len(pd.read_parquet(VAO / "don_vi_can_dich.parquet", columns=["ma_don_vi"]))
bao_cao = {"che_do": CHE_DO, "gpu": so_gpu, "dau_ra_cu": CU, "ngay": time.strftime("%Y-%m-%d %H:%M:%S")}
for k in ["A", "B"]:
    m = MO_HINH[k]
    n = sum(len(pd.read_parquet(f, columns=["ma_don_vi"])) for f in (RA / m).glob("lo_*.parquet"))
    nk = sum(len(pd.read_parquet(f, columns=["ma_don_vi"])) for f in (RA / f"kiwi_{m}").glob("lo_*.parquet"))
    the = json.loads((RA / f"the_{m}.json").read_text(encoding="utf-8")) if (RA / f"the_{m}.json").exists() else {}
    td = the.get("toc_do_don_vi_giay")
    uoc = f"{tong / td / 3600:.1f} giờ cho cả {tong:,} đơn vị" if td else "chưa đo được"
    print(f"{k} {m}: backend {the.get('backend')} | đã dịch {n:,} | đã chấm {nk:,} | tốc độ {td} đơn vị/giây | ước cả bộ: {uoc}")
    if the.get("loi_vllm"):
        print("   vLLM lỗi:", the["loi_vllm"][:300])
    if the.get("api", {}).get("chi_phi_uoc_usd") is not None and the.get("ngan_sach_usd"):
        a_ = the["api"]
        dich_ = max(1, the.get("da_dich_phien_nay", 0))
        print(f"   API trả phí: chi phí ước {a_['chi_phi_uoc_usd']} USD / ngân sách {the['ngan_sach_usd']} USD | {a_['yeu_cau']} yêu cầu, "
              f"{a_['thu_lai']} lần thử lại, {a_['loi_api']} lỗi | trung bình {a_['giay_cac_lan_goi'] / max(1, a_['yeu_cau']):.1f} giây/yêu cầu")
        print(f"   => ước cả bộ ({tong:,} đơn vị): khoảng {a_['chi_phi_uoc_usd'] / dich_ * tong:.2f} USD (giá giờ cao điểm, ước dư)")
    bao_cao[m] = {"da_dich": n, "da_cham": nk, "the": the, "uoc_ca_bo": uoc}
(RA / "the_kaggle.json").write_text(json.dumps(bao_cao, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("\nCác file đầu ra (tab Output):")
for f in sorted(RA.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(RA.parent)}  {f.stat().st_size:,} byte")